# W04-2 건물 속성 모으기 — 높이·층수·용도·사용승인일

<a href="https://colab.research.google.com/github/kloud80/dataminig_cau/blob/main/%EC%A4%91%EC%95%99%EB%8C%8026%ED%8A%B9%EA%B0%95/notebooks/w04_colab/W04_2%20%EA%B1%B4%EB%AC%BC%20%EC%86%8D%EC%84%B1%20%EB%AA%A8%EC%9C%BC%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1번에서 **바닥 윤곽**을 얻었다. 3D 매스를 세우려면 이제 **높이**가 필요하다.
높이는 사진에서 재지 않는다. **공공 장부**에서 읽는다. 그런데 장부는 여러 권이고, 서로 다른 말을 할 때가 있다.

| 단계 | 하는 일 | 파이프라인 스테이지 |
|---|---|---|
| 1 | PNU(필지 고유번호)를 쪼개 대장 조회 코드 만들기 | `sources/ledger.py` |
| 2 | 건축물대장 표제부 (건축HUB, 키 선택) | `s03_prior` |
| 3 | 도로명주소 검색 (키 선택) | `s03_prior` (juso) |
| 4 | GIS건물통합정보 속성과 대조 | `s02_target` |
| 5 | **높이 우선순위** 대장 → GIS → 층수×표준층고 | `s03_prior.choose_height` |
| 6 | **층고 검사** — 94.13 m ÷ 15층 = 6.3 m? | `s03_prior` · `s20` 경고 |

> 키가 없으면 실습 데이터(`s02_target.json`, `s03_prior.json`)에 저장된 **실제 응답**을 쓴다. 결과는 같다.
> 결과물: `out/prior.json`, `out/w04_2_height_check.png`

## 1. 준비

In [ ]:
# ── 0. 실습 데이터 확인 ─────────────────────────────────────────
# 이 노트북은 옆에 data/ipark/ 폴더가 있다고 가정한다 (아이파크타워 실제 파이프라인 결과).
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/kloud80/dataminig_cau.git"   # 강의 저장소 (필요한 폴더만 받는다)
REPO_DIR = "중앙대26특강/notebooks/w04_colab"

DATA = Path('data/ipark')
if not DATA.exists() and REPO_URL:
    print('실습 데이터를 내려받는 중...')
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, 'w04_repo'], check=False)
    subprocess.run(['git', '-C', 'w04_repo', 'sparse-checkout', 'set', REPO_DIR], check=False)
    hits = sorted(Path('w04_repo').glob('**/data/ipark'))
    if hits:
        os.chdir(hits[0].parent.parent)        # data/ 의 부모 폴더로 이동
if not DATA.exists():
    raise FileNotFoundError(
        'data/ipark/ 폴더가 없습니다.\n'
        '  1) 수업 저장소를 clone 하고 이 노트북을 w04_colab 폴더에서 여세요.\n'
        '  2) Colab 이라면 네트워크를 확인하고 이 셀을 다시 실행하세요.\n'
        '  (또는 data/ipark 폴더를 왼쪽 파일 탭에 그대로 올려도 됩니다)')

OUT = Path('out'); OUT.mkdir(exist_ok=True)
print('작업 폴더', Path.cwd())
print('실습 데이터', sorted(p.name for p in DATA.iterdir()))

In [ ]:
import requests
from urllib.parse import unquote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.font_manager as fm
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

OFFLINE = False      # True 면 키가 있어도 네트워크를 쓰지 않는다

In [ ]:
# ── API 키 (전부 선택) ──────────────────────────────────────────
# Colab: 왼쪽 열쇠 아이콘(보안 비밀)에 이름 그대로 등록하고 '노트북 액세스'를 켠다.
# 로컬: 환경변수로 넣는다. 키 값은 화면에 절대 찍지 않는다.
def get_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name)
    return v.strip() if v else None

_SECRETS = []
def secret(name):
    v = get_secret(name)
    if v:
        _SECRETS.append(v)
    print(f'{name:15s}', '있음' if v else '없음')
    return v

def scrub(e):
    """오류 메시지에 URL 째로 키가 섞여 나오는 일이 있다 → 지우고 보여 준다."""
    s = str(e)
    for v in _SECRETS:
        s = s.replace(v, '***')
    return s[:160]

DATA_GO_KR_KEY = secret('DATA_GO_KR_KEY')   # 공공데이터포털 — 건축HUB 건축물대장정보 서비스
JUSO_KEY       = secret('JUSO_KEY')         # 행정안전부 도로명주소 검색 API (business.juso.go.kr)

## 2. 장부는 여러 권이다

| 출처 | 무엇 | 갱신 | 키 |
|---|---|---|---|
| **건축물대장 표제부** (건축HUB) | 동별 높이·층수·용도·구조·사용승인일·면적 | 인허가 때마다 | 공공데이터포털 |
| **GIS건물통합정보** (VWorld NED) | 건물 윤곽 + 대장 속성 사본 | 분기~연 | VWorld (지금 차단) |
| **도로명주소** (행안부) | 건물명·도로명·건물관리번호·출입구 | **월** | juso.go.kr |
| OpenStreetMap | 윤곽, 가끔 높이·층수 | 수시 (자원봉사) | 없음 |

이 장부들을 잇는 열쇠가 **PNU** (필지 고유번호, 19자리)다. 1번의 대상 건물을 확정할 때(s02) GIS 속성에 함께 들어 있었다.

In [ ]:
s02 = json.load(open(DATA / 's02_target.json', encoding='utf-8'))
s03 = json.load(open(DATA / 's03_prior.json', encoding='utf-8'))
cand = s02['candidate']
PNU = cand['pnu']

def pnu_to_ledger_params(pnu):
    """PNU = 시군구(5) + 법정동(5) + 대지구분(1: 1=일반, 2=산) + 본번(4) + 부번(4)"""
    return {'sigunguCd': pnu[:5], 'bjdongCd': pnu[5:10],
            'platGbCd': '0' if pnu[10] == '1' else '1',     # 대장 API 는 0=대지, 1=산 (PNU 와 코드가 다르다!)
            'bun': pnu[11:15], 'ji': pnu[15:19]}

print('건물 :', cand['name'], '/ PNU', PNU)
print('  시군구', PNU[:5], '| 법정동', PNU[5:10], '| 대지구분', PNU[10], '| 본번', PNU[11:15], '| 부번', PNU[15:19])
LEDGER_PARAMS = pnu_to_ledger_params(PNU)
print('대장 조회 코드 →', LEDGER_PARAMS)

## 3. 건축물대장 표제부

건축HUB `getBrTitleInfo` 는 **필지 하나에 있는 모든 동**을 돌려준다. 아파트 단지라면 수십 줄이다.
파이프라인은 동 이름이 맞는 줄을 고르고, 없으면 **주건축물 중 연면적이 가장 큰 동**을 고른다.

> 공공데이터포털 키는 `Encoding` / `Decoding` 두 가지로 준다. `requests` 는 알아서 인코딩하므로 **Decoding 키**를 쓴다.
> 실수로 Encoding 키(`%2B` 같은 글자가 섞인 것)를 넣어도 되게 한 번 풀어 준다.

In [ ]:
LEDGER_URL = 'https://apis.data.go.kr/1613000/BldRgstHubService/getBrTitleInfo'

def fetch_title(params):
    key = unquote(DATA_GO_KR_KEY) if '%' in DATA_GO_KR_KEY else DATA_GO_KR_KEY
    r = requests.get(LEDGER_URL, timeout=20,
                     params={'serviceKey': key, 'numOfRows': 100, 'pageNo': 1, '_type': 'json', **params})
    r.raise_for_status()
    items = r.json()['response']['body']['items']
    items = items.get('item', []) if isinstance(items, dict) else []
    return items if isinstance(items, list) else [items]

def pick_title(items, dong_name):
    if dong_name:
        for it in items:
            if (it.get('dongNm') or '').strip() and it['dongNm'].strip() in dong_name:
                return it
    mains = [it for it in items if str(it.get('mainAtchGbCd', '0')) == '0'] or items
    return max(mains, key=lambda it: float(it.get('totArea') or 0))

ledger, ledger_source = None, None
if DATA_GO_KR_KEY and not OFFLINE:
    try:
        items = fetch_title(LEDGER_PARAMS)
        print(f'표제부 {len(items)}줄 받음')
        if items:
            ledger, ledger_source = pick_title(items, cand.get('dong_name')), 'api'
    except Exception as e:
        print('건축HUB 호출 실패:', scrub(e))
if ledger is None:
    ledger, ledger_source = s03['ledger'], 'staged_s03'
    print('키가 없거나 호출 실패 → 실습 데이터의 표제부 응답을 쓴다 (실제 실행 때 받은 것)')

FIELDS = {'bldNm': '건물명', 'dongNm': '동명', 'mainPurpsCdNm': '주용도', 'strctCdNm': '구조',
          'grndFlrCnt': '지상 층수', 'ugrndFlrCnt': '지하 층수', 'heit': '높이(m)',
          'archArea': '건축면적(㎡)', 'totArea': '연면적(㎡)', 'platArea': '대지면적(㎡)',
          'bcRat': '건폐율(%)', 'vlRat': '용적률(%)', 'useAprDay': '사용승인일',
          'rideUseElvtCnt': '승용 승강기', 'roofCdNm': '지붕'}
display(pd.DataFrame({'항목': list(FIELDS.values()), '값': [ledger.get(k) for k in FIELDS]}).set_index('항목'))

## 4. 도로명주소 — 월 단위로 갱신되는 장부

대장·GIS 는 갱신이 느리다. 도로명주소 DB 는 **매달** 갱신된다. 건물 이름이 바뀌었거나 새 건물이면 여기가 가장 최신이다.
파이프라인은 여기서 **건물관리번호**(안정적인 건물 키)와, 승인이 나면 **출입구 좌표**까지 받는다 — 어느 벽이 정면인지 사진 없이 알 수 있다.
(출입구 좌표 API 는 아직 승인 대기라 이번 실행에는 없다.)

In [ ]:
ADDRESS = '서울특별시 강남구 영동대로 520'
juso, juso_source = None, None
if JUSO_KEY and not OFFLINE:
    try:
        r = requests.get('https://business.juso.go.kr/addrlink/addrLinkApi.do', timeout=15,
                         params={'confmKey': JUSO_KEY, 'currentPage': 1, 'countPerPage': 1,
                                 'keyword': ADDRESS, 'resultType': 'json'})
        res = r.json()['results']
        if res['common']['errorCode'] != '0':
            print('도로명주소 오류:', res['common']['errorMessage'])
        elif res['juso']:
            it = res['juso'][0]
            juso = {'bd_nm': it.get('bdNm'), 'road_addr': it.get('roadAddr'), 'jibun_addr': it.get('jibunAddr'),
                    'zip_no': it.get('zipNo'), 'bd_mgt_sn': it.get('bdMgtSn'), 'road_name': it.get('rn')}
            juso_source = 'api'
    except Exception as e:
        print('도로명주소 호출 실패:', scrub(e))
if juso is None:
    juso, juso_source = s03['juso'], 'staged_s03'
    print('키가 없거나 호출 실패 → 실습 데이터의 도로명주소 응답을 쓴다')
for k in ['bd_nm', 'road_addr', 'jibun_addr', 'zip_no', 'bd_mgt_sn']:
    print(f'{k:11s}', juso.get(k))

## 5. GIS 속성과 대조 — 두 장부가 같은 말을 하는가

GIS건물통합정보에도 높이·층수·용도가 붙어 있다 (1번 노트북의 s02 후보 속성).
대장과 나란히 놓아 보자.

In [ ]:
raw = cand['raw']
pairs = [('높이(m)', raw.get('hg'), ledger.get('heit')),
         ('지상 층수', raw.get('ground_floor_co'), ledger.get('grndFlrCnt')),
         ('지하 층수', raw.get('undgrnd_floor_co'), ledger.get('ugrndFlrCnt')),
         ('용도 코드', raw.get('buld_prpos_code'), ledger.get('mainPurpsCd')),
         ('건축면적(㎡)', raw.get('ar'), ledger.get('archArea')),
         ('연면적(㎡)', raw.get('totar'), ledger.get('totArea')),
         ('사용승인일', raw.get('use_confm_de'), ledger.get('useAprDay'))]
norm = lambda v: str(v).replace('-', '').strip()
tbl = pd.DataFrame([{'항목': a, 'GIS건물통합정보': g, '건축물대장': l, '같은가': norm(g) == norm(l)} for a, g, l in pairs])
display(tbl.set_index('항목'))

**전부 같다.** 좋은 소식일까?

GIS건물통합정보의 속성은 **건축물대장(세움터)에서 옮겨 온 사본**이다. 두 장부가 같은 말을 하는 것은
"두 사람이 확인했다" 가 아니라 **"한 사람의 말을 두 번 들었다"** 에 가깝다.
대장이 틀렸으면 GIS 도 똑같이 틀린다. (1번에서 본 OSM 의 94 m · 15층 태그도 아마 같은 출처다.)

## 6. 높이 우선순위 — 파이프라인의 규칙

장부가 여러 권이면 **누구 말을 먼저 믿을지** 정해 둬야 한다. 파이프라인 s03 의 규칙이다.

1. **건축물대장** 의 높이
2. 없으면 **GIS** 의 높이
3. 그것도 없으면 **층수 × 표준층고** — 용도별 표준층고 (주거 2.9 m, 업무·상업 3.8 m, 상가 1층 4.5 m, 기타 3.3 m)

그리고 **상식 검사**: 층당 높이가 1.8 m 미만이거나 8 m 를 넘으면 장부 오류로 보고 3번(층수×표준층고)으로 내려간다.
고른 높이는 다시 층별로 나눈다 — 1층은 조금 높게, 나머지는 같게. 그게 2.4~6.0 m 범위를 벗어나면 그냥 똑같이 나눈다.

In [ ]:
FLOOR_H = {'residential': 2.9, 'commercial': 3.8, 'ground_commercial': 4.5, 'default': 3.3}
RESIDENTIAL = {'01', '02'}                          # 단독·공동주택
GROUND_COMMERCIAL = {'03', '04', '07', '14', '15'}  # 근린생활·판매·업무·숙박
MIN_HEIGHT = 3.0

def standard_floor_heights(n, use_code, total=None):
    grp = (use_code or '')[:2]
    typical = (FLOOR_H['residential'] if grp in RESIDENTIAL else
               FLOOR_H['commercial'] if grp in GROUND_COMMERCIAL else FLOOR_H['default'])
    ground = FLOOR_H['ground_commercial'] if grp in GROUND_COMMERCIAL else typical
    if n <= 1:
        return [total or ground]
    if total:
        rest = (total - ground) / (n - 1)
        if 2.4 <= rest <= 6.0:
            return [ground] + [rest] * (n - 1)
        return [total / n] * n
    return [ground] + [typical] * (n - 1)

def decide_height(ledger_h, gis_h, floors, use_code):
    """반환: 높이, 출처, 층수, 층고 목록, 후보, 경고"""
    cands, warn = [], []
    if ledger_h and float(ledger_h) > 0:
        cands.append(('ledger', float(ledger_h)))
    if gis_h and float(gis_h) > 0:
        cands.append(('gis_building', float(gis_h)))
    floors = int(floors or 0)
    floors_known = floors > 0               # 0층은 '1층' 이 아니라 '모름'
    floors = max(floors, 1)
    cands.append(('floors_x_standard', round(sum(standard_floor_heights(floors, use_code)), 2)))
    order = ['ledger', 'gis_building', 'floors_x_standard']
    src, h = min(cands, key=lambda c: order.index(c[0]))
    h = max(h, MIN_HEIGHT)
    if not floors_known and src != 'floors_x_standard':
        floors = max(1, round(h / FLOOR_H['default']))
    if not (1.8 <= h / floors <= 8.0):
        warn.append(f'높이 {h:.1f} m / {floors}층 = 층당 {h / floors:.1f} m 비정상 → 층수×표준층고')
        src, h = 'floors_x_standard', dict(cands)['floors_x_standard']
    return {'height_m': round(h, 2), 'height_source': src, 'floors_above': floors,
            'floor_heights': standard_floor_heights(floors, use_code, h),
            'height_candidates': [{'source': s, 'value': v} for s, v in cands], 'warnings': warn}

prior = decide_height(ledger.get('heit'), cand.get('height_m'), ledger.get('grndFlrCnt') or cand.get('floors_above'),
                      ledger.get('mainPurpsCd') or cand.get('use_code'))
display(pd.DataFrame(prior['height_candidates']).rename(columns={'source': '후보', 'value': '높이(m)'}))
print(f"채택: {prior['height_m']} m  (출처 {prior['height_source']}, {prior['floors_above']}층)")
print('층고:', [round(x, 2) for x in prior['floor_heights'][:3]], '... 전부', len(prior['floor_heights']), '개')
print('경고:', prior['warnings'] or '없음')

# 실제 파이프라인 결과와 같은지
same = (abs(prior['height_m'] - s03['height_m']) < 1e-6 and prior['height_source'] == s03['height_source']
        and np.allclose(prior['floor_heights'], s03['floor_heights']))
print('\n실제 실행(s03)과 같은가:', same)

### 규칙을 흔들어 보기
장부가 비어 있거나 이상할 때 이 함수가 어떻게 움직이는지 본다.

In [ ]:
cases = {'지금 (대장·GIS 모두 있음)': (94.13, 94.13, 15, '14000'),
         '대장 없음, GIS 만':        (None, 94.13, 15, '14000'),
         '높이 모두 없음, 층수만':     (None, None, 15, '14000'),
         '같은 15층이 주거라면':       (None, None, 15, '02000'),
         '층수 0 (모름)':            (94.13, None, 0, '14000'),
         '높이 150 m / 15층 (오타?)': (150, None, 15, '14000')}
rows = []
for name, args in cases.items():
    p = decide_height(*args)
    rows.append({'경우': name, '높이(m)': p['height_m'], '출처': p['height_source'], '층수': p['floors_above'],
                 '1층 층고': round(p['floor_heights'][0], 2), '2층 이상 층고': round(p['floor_heights'][-1], 2),
                 '경고': '; '.join(p['warnings'])})
display(pd.DataFrame(rows).set_index('경우'))

## 7. 층고 검사 — 6.3 m 는 정상인가

아이파크타워의 대장: **지상 15층, 높이 94.13 m.** 나누면 **층당 6.3 m** 다.
보통 사무실 층고는 3.6~4.2 m 다. 6.3 m 면 한 층이 아파트 두 층 높이다.

파이프라인의 상식 검사(1.8~8.0 m)는 **통과**했다. 그래서 그대로 썼다.
그런데 뒷 단계(s20)에서 거리 사진을 판독한 층고를 더해 보니 이런 경고가 나왔다.

```
s20 경고: 층고 합(53.2m)이 건물 높이(94.1m)와 크게 달라 균등 분할로 대체
```

사진으로 읽은 층고를 더하면 **53.2 m** 인데 장부는 **94.1 m**. 1.77배 차이다.
파이프라인은 0.6~1.6배까지만 사진 값을 비례로 늘려 쓰고, 그 밖이면 사진 값을 버리고 장부를 따른다 — 이번에는 장부를 따랐다.

In [ ]:
H, N = prior['height_m'], prior['floors_above']
PHOTO_SUM = 53.2      # 파이프라인 s20 로그 (거리 사진 판독 층고 합)

fig, ax = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={'width_ratios': [1.25, 1]})

per = {'주거 표준': 2.9, '기타 표준': 3.3, '업무·상업 표준': 3.8, '상가 1층 표준': 4.5,
       '사진 판독 (53.2 m ÷ 15)': PHOTO_SUM / N, f'대장 ({H} m ÷ {N})': H / N}
cols = ['#B8C4D6'] * 4 + ['#4C9F70', '#E8452C']
ax[0].axvspan(1.8, 8.0, color='#FFE9A8', alpha=0.5, label='파이프라인 허용 범위 1.8~8.0 m')
ax[0].barh(list(per), list(per.values()), color=cols)
for y, v in enumerate(per.values()):
    ax[0].text(v + 0.08, y, f'{v:.2f} m', va='center', fontsize=12)
ax[0].set_xlim(0, 9); ax[0].set_xlabel('층당 높이 (m)'); ax[0].legend(loc='lower right')
ax[0].set_title('층당 높이 — 상식 검사는 통과했다', fontsize=14)

tot = {'장부 (대장 = GIS = OSM)': H, '층수 × 표준층고': dict((c['source'], c['value']) for c in prior['height_candidates'])['floors_x_standard'],
       '거리 사진 판독 합': PHOTO_SUM}
ax[1].bar(list(tot), list(tot.values()), color=['#E8452C', '#B8C4D6', '#4C9F70'], width=0.55)
for x, v in enumerate(tot.values()):
    ax[1].text(x, v + 1.5, f'{v:.1f} m', ha='center', fontsize=13, weight='bold')
ax[1].set_ylim(0, 110); ax[1].set_ylabel('건물 높이 (m)')
ax[1].set_title(f'같은 건물, 세 개의 높이 ({N}층)', fontsize=14)
plt.tight_layout(); plt.savefig(OUT / 'w04_2_height_check.png', dpi=150); plt.show()

### 층별 면적은 무엇을 말하나
대장의 **층별개요**에는 층마다 바닥면적이 있다. 저층부만 크고 위가 좁은 건물이면 "저층부 층고가 높다" 는 설명이 될 수도 있다.

In [ ]:
fa = {int(k): v for k, v in (s03.get('floor_areas') or {}).items()}
if fa:
    plt.figure(figsize=(10, 3.2))
    plt.bar(list(fa), list(fa.values()), color='#7A8CA8')
    plt.xticks(list(fa)); plt.xlabel('지상 층'); plt.ylabel('바닥면적 (㎡)')
    plt.title('층별 바닥면적 (건축물대장 층별개요)'); plt.show()
    v = np.array(list(fa.values()))
    print(f'최소 {v.min():,.0f} ㎡ · 최대 {v.max():,.0f} ㎡ · 15개 층 합 {v.sum():,.0f} ㎡'
          f'  (대장의 용적률 산정 연면적 {ledger.get("vlRatEstmTotArea"):,} ㎡)')

층마다 면적이 거의 같다 (약 1,150~1,280 ㎡). 위로 좁아지는 탑이 아니다. **층별 면적으로는 6.3 m 를 설명할 수 없다.**

### 그럼 누가 틀렸나 — 가설을 세워 보자
* **(가) 높이에 옥탑·장식 구조물이 들어갔다** — 대장의 높이는 건축법상 높이다. 옥상 구조물 일부가 포함되기도 한다.
* **(나) 정말 층고가 높은 건물이다** — 업무시설 중에는 층고를 크게 잡는 경우가 있다.
* **(다) 층수나 높이 중 하나가 잘못 입력됐다** — 장부도 사람이 친다.
* **(라) 사진 판독이 틀렸다** — 판독 모델이 고층부를 다 못 봤거나 층 경계를 잘못 셌다.

어느 쪽인지는 **이 노트북만으로는 알 수 없다.** 그게 핵심이다.

> **데이터 품질 교훈**
> 1. 상식 검사를 **통과했다고 맞는 것은 아니다.** 1.8~8.0 m 는 "불가능" 만 걸러 낸다. "이상함" 은 거르지 못한다.
> 2. 같은 값이 세 군데서 나와도, **출처가 하나면 근거는 하나**다 (대장 → GIS → OSM).
> 3. **독립된 측정**(사진 판독)이 붙어야 이상함이 드러난다. 그리고 그때 누구를 믿을지는 규칙으로 정해 둔다 — 그 규칙도 틀릴 수 있다.
> 4. 그래서 파이프라인은 값을 고칠 때 **경고를 남긴다.** 조용히 고치면 아무도 다시 보지 않는다.

## 8. 용도와 사용승인일

용도 코드 앞 두 자리가 큰 분류다. 파이프라인은 이것으로 표준층고와 1층이 상가인지를 정한다.
사용승인일은 건물의 나이다. (원래는 "사용승인일보다 오래된 거리 사진은 쓰지 않는다" 에 쓰려 했지만,
로드뷰 촬영일을 얻을 수 없어 아직 쓰지 못한다.)

In [ ]:
USE_GROUPS = {'01': '단독주택', '02': '공동주택', '03': '제1종근린생활시설', '04': '제2종근린생활시설',
              '05': '문화및집회시설', '06': '종교시설', '07': '판매시설', '08': '운수시설', '09': '의료시설',
              '10': '교육연구시설', '11': '노유자시설', '12': '수련시설', '13': '운동시설', '14': '업무시설',
              '15': '숙박시설', '16': '위락시설', '17': '공장', '18': '창고시설'}
use_code = ledger.get('mainPurpsCd') or cand.get('use_code')
use_name = USE_GROUPS.get(use_code[:2], '기타')
apr = str(ledger.get('useAprDay') or cand.get('approval_date') or '').replace('-', '')
approval = f'{apr[:4]}-{apr[4:6]}-{apr[6:8]}' if len(apr) == 8 else None
age = 2026 - int(apr[:4]) if approval else None
grp = use_code[:2]
print(f'용도   {use_code} → {use_name}')
print(f'  표준층고 그룹: {"주거" if grp in RESIDENTIAL else "업무·상업 (1층 4.5 m)" if grp in GROUND_COMMERCIAL else "기타"}')
print(f'사용승인일 {approval}  → 2026년 기준 약 {age}년 된 건물')

## 9. 저장 — `out/prior.json`

In [ ]:
warnings = prior['warnings'] + [
    f"층당 {H / N:.2f} m — 상식 범위(1.8~8.0 m)는 통과했으나 업무시설 통상값(약 3.8 m)의 {H / N / 3.8:.1f}배",
    f"s20: 사진 판독 층고 합 {PHOTO_SUM} m vs 장부 {H} m → 장부를 따름 (확인 필요)"]
out = {
    'building_id': cand['building_id'], 'pnu': PNU, 'name': ledger.get('bldNm') or cand.get('name'),
    'road_addr': juso.get('road_addr'), 'bd_mgt_sn': juso.get('bd_mgt_sn'),
    'use_code': use_code, 'use_name': use_name, 'approval_date': approval,
    'floors_above': prior['floors_above'], 'floors_below': ledger.get('ugrndFlrCnt'),
    'height_m': prior['height_m'], 'height_source': prior['height_source'],
    'height_candidates': prior['height_candidates'],
    'floor_heights': [round(x, 4) for x in prior['floor_heights']],
    'per_floor_m': round(H / N, 3),
    'arch_area_m2': ledger.get('archArea'), 'total_area_m2': ledger.get('totArea'),
    'warnings': warnings,
    'sources': {'ledger': ledger_source, 'juso': juso_source, 'gis': 'staged_s02'},
}
json.dump(out, open(OUT / 'prior.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

print('── W04-2 요약 ───────────────────────────')
print(f"건물       {out['name']}  ({out['road_addr']})")
print(f"용도       {use_name} ({use_code}) · 사용승인 {approval}")
print(f"층수       지상 {N} / 지하 {out['floors_below']}")
print(f"높이       {H} m  ← {prior['height_source']}  (후보: " +
      ', '.join(f"{c['source']} {c['value']}" for c in prior['height_candidates']) + ')')
print(f"층당 높이  {H / N:.2f} m   ⚠ 데이터 품질 확인 필요")
print(f"출처       대장 {ledger_source} · 도로명주소 {juso_source}")
print('저장      ', ', '.join(sorted(p.name for p in OUT.glob('*'))))

## 해 볼 것

1. 6번 `cases` 에 **3층 단독주택, 높이 0** 을 넣어 보자. 표준층고 규칙은 어떤 높이를 주는가? 실제 단독주택과 비교하면?
2. 상식 범위 1.8~8.0 m 를 **2.4~5.0 m** 로 좁히면 아이파크타워는 어떻게 되는가? 그 대신 무엇이 망가질 수 있을까 (체육관·공장·예배당)?
3. 7번의 가설 (가)~(라) 중 하나를 골라, **어떤 데이터가 있으면 확인할 수 있는지** 적어 보자. (힌트: 층별개요의 층 구분, 위성사진 그림자 길이, 다른 각도의 사진, 건축 도면 공개 자료)
4. 공공데이터포털에서 건축HUB 키를 발급받아 Colab 보안 비밀에 `DATA_GO_KR_KEY` 로 넣고 다시 돌려 보자. 실습 데이터와 같은 값이 오는가?
5. 우리 동네 아파트 단지 하나의 PNU 로 표제부를 받아 보자. 동이 몇 줄 나오는가? `pick_title` 은 어느 동을 고르는가 — 그게 원하는 동인가?

### 참고
* 국토교통부 건축HUB 건축물대장정보 서비스 — `apis.data.go.kr/1613000/BldRgstHubService` (공공데이터포털)
* 행정안전부 도로명주소 검색 API — business.juso.go.kr
* addr2blend 파이프라인 (사내 연구용) — `sources/ledger.py`, `sources/juso.py`, `stages/s03_prior.py`